# Writing Your Own Functions

As scripts become longer, you will often repeat the same sequence of steps. A function lets you give that sequence a name and reuse it whenever you need it.

For example, base R does not include a dedicated function for calculating the standard error of a mean. Rather than repeatedly writing the calculation yourself, you can create a function for it.

The standard error is calculated as:

$$
SE = \frac{s}{\sqrt{n}}
$$

where $s$ is the sample standard deviation and $n$ is the number of observations.

In R, you create a function with `function()`. List the inputs it needs inside the parentheses, write the instructions inside curly brackets `{}`, and call the function using its name followed by parentheses.



## A simple function

Here is a function that calculates the standard error of a numeric vector:

```r
standard_error <- function(x) {
  sd(x) / sqrt(length(x))
}
```

Running this code defines the function and stores it in an object called `standard_error`. The function has one argument: `x`.

To use it, call the function with a vector of values:

```r
scores <- c(4, 6, 5, 7, 8)

standard_error(scores)
```

```text
[1] 0.7071068
```

Inside the function:

- R receives the supplied vector and assigns it to `x`.
- It calculates the standard deviation using `sd(x)`.
- It divides this by the square root of the number of values in `x`.
- The final expression becomes the function’s result.

A function does not run when you define it. It runs only when you call it.



## Returning values

Every R function returns a value. By default, R returns the result of the final expression evaluated within the function body.

In this function, the calculation is the final expression, so it is returned automatically:

```r
standard_error <- function(x) {
  sd(x) / sqrt(length(x))
}
```

You can make the calculation easier to read by storing it in an object first:

```r
standard_error <- function(x) {
  se <- sd(x) / sqrt(length(x))
  se
}
```

```r
standard_error(scores)
```

```text
[1] 0.7071068
```

The final line, `se`, is returned.

You can also use `return()` to state explicitly what the function should return. This is especially helpful when a function needs to stop early under particular conditions.



## A more robust standard-error function

A standard error cannot be calculated for non-numeric data or for a vector with fewer than two observations. We can use `return()` to stop the function early in these cases.

We can also add two optional arguments:

- `na.rm`, which controls whether missing values are removed before calculation.
- `interval`, which controls whether the function returns the standard error alone or the mean plus or minus one standard error.

```r
standard_error <- function(x, na.rm = FALSE, interval = FALSE) {
  
  # Stop early if x is not numeric
  if (!is.numeric(x)) {
    return(NA)
  }
  
  # Optionally remove missing values
  if (na.rm) {
    x <- x[!is.na(x)]
  }
  
  # Stop early if fewer than two values remain
  if (length(x) < 2) {
    return(NA)
  }
  
  # Calculate the standard error
  se <- sd(x) / sqrt(length(x))
  
  # Return the mean and one standard error either side, if requested
  if (interval) {
    mean_value <- mean(x)
    
    return(c(
      lower = mean_value - se,
      mean = mean_value,
      upper = mean_value + se
    ))
  }
  
  # Otherwise, return the standard error
  se
}
```

By default, the function returns only the standard error:

```r
scores <- c(4, 6, 5, 7, 8)

standard_error(scores)
```

```text
[1] 0.7071068
```

Set `interval = TRUE` to return the mean and a range of one standard error either side:

```r
standard_error(scores, interval = TRUE)
```

```text
   lower     mean    upper 
5.292893 6.000000 6.707107 
```



## Default argument values

The arguments `na.rm = FALSE` and `interval = FALSE` are **default argument values**. They mean that, unless specified otherwise, the function:

- does not remove missing values; and
- returns only the standard error.

This keeps the most common use of the function short:

```r
standard_error(scores)
```

However, you can change the defaults when needed. For example, if the vector contains a missing value, set `na.rm = TRUE`:

```r
scores_with_missing <- c(4, 6, 5, NA, 8)

standard_error(scores_with_missing, na.rm = TRUE)
```

```text
[1] 0.8539126
```

You can combine named arguments to remove missing values and return the mean plus or minus one standard error:

```r
standard_error(scores_with_missing, na.rm = TRUE, interval = TRUE)
```

```text
   lower     mean    upper 
4.896087 5.750000 6.603913 
```

The function also returns `NA` when the input is unsuitable.

For example, a standard error cannot be calculated from a single value:

```r
standard_error(5)
```

```text
[1] NA
```

It also cannot be calculated from non-numeric data:

```r
standard_error(c("low", "medium", "high"))
```

```text
[1] NA
```

Here, `return()` ends the function immediately when:

- `x` is not numeric;
- fewer than two observations are available; or
- `interval = TRUE`, after the interval has been created.

Default values make functions more flexible while keeping common calls short. Arguments with default values should usually appear after arguments without defaults.
